In [ ]:
import os
import re
import sys
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
from os.path import join as pjoin
from natsort import natsorted

sys.path.append('../../')
import circletrack_neural as ctn
import circletrack_behavior as ctb
import place_cells as pc

project_dir = 'MultiCon_Imaging'
experiment_dir = 'MultiCon_Imaging7'
sdata_path = f'../../../{project_dir}/{experiment_dir}/output/aligned_minian/'
minian_path = os.path.abspath(f'../../../{project_dir}/{experiment_dir}/minian_results')
output_path = os.path.abspath(f'../../../{project_dir}/{experiment_dir}/output')
mouse_info =  pd.read_csv(os.path.abspath(f'../../../{project_dir}/{experiment_dir}/maze_yml/mouse_info.csv'))
mouse_list = ['mc62', 'mc63', 'mc64', 'mc65']
session_type = 'YrA'

In [ ]:
for mouse in mouse_list:
    mpath = os.path.abspath(pjoin(minian_path, mouse))
    for idx, session in tqdm(enumerate(natsorted(os.listdir(mpath)))):
        print(f'Aligning {session}...')
        
        dpath = pjoin(mpath, session)
        timestamp = os.listdir(dpath)[0] ## minian timestamp is first folder
        date = re.search('20[0-2][0-9]_[0-9]+_[0-9]+', dpath)[0]
        data_path = pjoin(dpath, f'{timestamp}/minian')
        spikes = ctn.open_minian(data_path)[session_type] ## select spike matrix
        sdata = xr.open_dataset(pjoin(sdata_path, f'{mouse}/S/{mouse}_S_{idx + 1}.nc'))['S']
        passed_qc = sdata['passed_qc'].values

        spath = pjoin(output_path, f'aligned_minian/{mouse}/{session_type}')
        if not os.path.exists(spath):
            os.makedirs(spath)

        file_name = natsorted(os.listdir(pjoin(output_path, f'behav/{mouse}')))[idx]
        behav = pd.read_feather(pjoin(output_path, f'behav/{mouse}/{file_name}'))
        save_path = pjoin(spath, f'{mouse}_{session_type}_{idx+1}.nc')
        ## Select session
        imaging_session = spikes.copy()
        ## Align calcium and behavior based on unix timestamps
        cropped_calc, aligned_behav = ctn.align_calcium_behavior(imaging_session, behav)
        ## Assign behavior data as coordinates
        cropped_calc = cropped_calc.assign_coords(behav_t=('frame', aligned_behav['t']),
                                                behav_frame=('frame', aligned_behav['frame']),
                                                x=('frame', aligned_behav['x']),
                                                y=('frame', aligned_behav['y']),
                                                a_pos=('frame', aligned_behav['a_pos']),
                                                correct_dir=('frame', aligned_behav['correct_dir']),
                                                lick_port=('frame', aligned_behav['lick_port']),
                                                water=('frame', aligned_behav['water']),
                                                trials=('frame', aligned_behav['trials']),
                                                lin_position=('frame', aligned_behav['lin_position']),
                                                probe=('frame', aligned_behav['probe']))
        cropped_calc = cropped_calc.assign_coords(passed_qc=('unit_id', passed_qc))
        cropped_calc = cropped_calc.assign_attrs(dict(animal=aligned_behav.loc[0, 'animal'],
                                                    session=aligned_behav.loc[0, 'session'],
                                                    session_two=aligned_behav.loc[0, 'session_two'],
                                                    cohort=aligned_behav.loc[0, 'cohort'],
                                                    reward_one=aligned_behav.loc[0, 'reward_one'],
                                                    reward_two=aligned_behav.loc[0, 'reward_two'],
                                                    maze=aligned_behav.loc[0, 'maze'],
                                                    date=date,
                                                    timestamp=timestamp,
                                                    sex=mouse_info['Sex'][mouse_info['Mouse'] == mouse].values[0],
                                                    group=mouse_info['Group'][mouse_info['Mouse'] == mouse].values[0]))
        cropped_calc = cropped_calc.reset_coords(names='animal', drop=True)
        ## Remove the first five seconds of the behavior, since the mouse is getting dropped into the maze then.
        cropped_calc = cropped_calc[:, cropped_calc['behav_t'] >= 5]
        # Save cropped_calc as a netcdf file
        cropped_calc.to_netcdf(save_path)